[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navidsharifi/optora-reproductions/blob/main/notebooks/reproductions.ipynb)

# Optora reproductions

Reproductions of published distributionally robust optimization papers with [Optora](https://github.com/navidsharifi/optora). This notebook is generated by `python -m tools.notebooks` from [`papers/`](https://github.com/navidsharifi/optora-reproductions/blob/main/papers) and [`results/`](https://github.com/navidsharifi/optora-reproductions/blob/main/results) of [navidsharifi/optora-reproductions](https://github.com/navidsharifi/optora-reproductions), so changes belong in those files, not here.

Papers in this notebook

- `duchi2019_variance_regularization`


## Recorded results

No claims have been reproduced yet.

These are the committed records of the full runs. A claim is reproduced when the mean over seeds lies within the tolerance that was fixed before the run.


## Setup

Choose a GPU runtime (Runtime, Change runtime type), then run the cell below once before rerunning any paper.


In [ ]:
# Clone (or update) the repository and install it with its pinned optora
# release. Colab already ships a CUDA build of PyTorch, which pip keeps.
![ -d /content/optora-reproductions ] || git clone --depth 1 https://github.com/navidsharifi/optora-reproductions.git /content/optora-reproductions
!git -C /content/optora-reproductions pull --ff-only --quiet
%cd /content/optora-reproductions
!pip install --quiet -e .

import optora
import torch
from IPython.display import Markdown, display

from optora_reproductions.results import read_results
from tools.results_table import render_table

print("optora", optora.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "none")


## Variance-based regularization with convex objectives

> Status. `in progress`

### Reference

John Duchi and Hongseok Namkoong. Variance-based regularization with
convex objectives. Journal of Machine Learning Research, 20(68):1-55,
2019. arXiv:1610.02581. An earlier version appeared at NeurIPS 2017.

Citation checked against the JMLR page on 2026-11-15. The authors' public
code (`hsnamkoong/robustopt`) holds projection routines only, not the
simulation reproduced here.

### Claim

We reproduce the simulation of Section 5.2. The paper minimizes the
robustly regularized risk

$$
R_n(\theta, \mathcal P_n) = \sup_{P} \left\{ \mathbb E_P[\ell(\theta; X)] :
D_\phi(P \,\|\, \hat P_n) \le \rho / n \right\}, \qquad \phi(t) = \tfrac12 (t - 1)^2,
$$

and compares its minimizer $\hat\theta^{\rm rob}$ with the empirical risk
minimizer $\hat\theta^{\rm erm}$ through the population risk
$R(\theta) = \tfrac12 \|\theta - v\|_2^2$.

| Claim id | Statement |
| --- | --- |
| c1 | The robust solution has lower population risk than ERM in every replication, for every sample size and noise level |
| c2 | For $B = 1$ and $n = 1000$ the robust solution recovers $v$, so its risk is zero up to the numerical precision of the solver |
| c3 | For $B = 1$ and $n = 100$, the one setting where the robust solution is not $v$, its risk is about 0.18 times the ERM risk |

The location, metric, paper value and tolerance of each claim are fixed
in `claims.py`, committed before the full run.

### Setup

| Item | Value |
| --- | --- |
| Loss | $\ell(\theta; x) = \tfrac12 \lVert\theta - v\rVert_2^2 + x^\top(\theta - v)$ with $d = 50$ |
| Data | $X \sim \mathrm{Uni}(\{-B, B\}^d)$, $B \in \{0.01, 0.1, 1, 10\}$, $n \in \{10^2, 5\cdot10^2, 10^3, 5\cdot10^3, 10^4\}$ |
| Domain | $\Theta = \{\theta : \lVert\theta\rVert_2 \le r\}$, $r = 10$, $v = \tfrac{r}{2\sqrt d}\mathbf 1$ |
| Radius | $\rho = \log(2/\delta) + d \log(2nDL)$ (Eq. 17), $\delta = 0.05$, $D = 2r$, $L = 3r + \sqrt d B$ |
| Optora components | `optora.ChiSquareAmbiguitySet` with radius $2\rho/n$, since Optora's chi-square divergence $\sum_i (q_i - p_i)^2 / p_i$ equals $2 D_\phi$ |
| Robust solver | Projected gradient descent with backtracking (Armijo) line search, as in Section 5.1, batched over replications |
| ERM solver | Closed form $\hat\theta^{\rm erm} = \Pi_\Theta(v - \bar X_n)$, the exact minimizer of the empirical risk over $\Theta$ |
| Replications | 1,200 per $(n, B)$ as in the paper, split into 3 seeds of 400 so each claim has a spread over seeds |
| Precision | float64 |
| Hardware | RTX 3050 Laptop 6 GB (full run), CPU (smoke run) |

Known discrepancy, recorded before any run. Every ERM entry of Table 1
with $B \ge 0.1$ equals $dB^2/(12n)$ to about 1%. The setup as stated gives
$\mathbb E[R(\hat\theta^{\rm erm})] = \tfrac12 \mathbb E\lVert\bar X_n\rVert_2^2 = dB^2/(2n)$,
six times larger. We run the setup as stated. Claims c1 and c3 do not
depend on this scale and c2 sits at the numerical floor.

### Run

```bash
python -m papers.duchi2019_variance_regularization.run --config papers/duchi2019_variance_regularization/configs/smoke.toml
python -m papers.duchi2019_variance_regularization.run --config papers/duchi2019_variance_regularization/configs/full.toml
```

### Result

See the results table in the root `README.md`, generated from
`results/duchi2019_variance_regularization.<claim_id>.json`.

### Findings

Pending the full run.


### Recorded results of `duchi2019_variance_regularization`

No claims have been reproduced yet.


### Rerun `duchi2019_variance_regularization` on Colab

The smoke configuration takes seconds. The full configuration repeats the recorded run and can take much longer.


In [ ]:
!python -m papers.duchi2019_variance_regularization.run --output-dir runs/colab --config papers/duchi2019_variance_regularization/configs/smoke.toml


In [ ]:
!python -m papers.duchi2019_variance_regularization.run --output-dir runs/colab --config papers/duchi2019_variance_regularization/configs/full.toml


In [ ]:
reruns = [r for r in read_results("runs/colab") if r.paper_id == "duchi2019_variance_regularization"]
display(Markdown(render_table(reruns)))


## All rerun results

Every claim rerun in this session.


In [ ]:
reruns = [r for r in read_results("runs/colab")]
display(Markdown(render_table(reruns)))
